In [1]:
import faiss
import numpy as np
import os
import torch
from langchain_ollama import ChatOllama
from langchain.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import Any
from uuid import uuid4
torch.cuda.is_available()

True

In [4]:
llm = ChatOllama(
    model="gemma4:latest"
)

In [5]:
class QueryEnhancerOutput(BaseModel):
    query : str = Field(default = "", description="This is for enhance user query")

class ExtractorOutput(BaseModel):
    entities : list[Any]
    relationships : list[Any]

# VectorRAG

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_docling.loader import DoclingLoader, ExportType, Document
from sentence_transformers import SentenceTransformer
import faiss

file_path = os.path.join("..", "assets", "example1.pdf")

def extraction(file_path):
    loader = DoclingLoader(
        file_path = file_path,
        export_type = ExportType.MARKDOWN
    )

    return loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def chunk_fn(docs : list[Document]) -> list[Document]:
    chunks = text_splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["order"] = i 
        chunk.metadata["id"] = uuid4()
    return chunks

embedding_model = SentenceTransformer("BAAI/bge-m3")

def embedding_fn(chunk_page_content : list[str]):
    return embedding_model.encode(chunk_page_content)

d = 1024 # depending on the embedding model dimension
#* IndexFlatIP stores normalized vector, but BAAI/bge-m3 output is a normalized vector
#* so, doesn't need to normalize the output vector again
index = faiss.IndexFlatIP(d)

def store_chunks(embedded_chunks : np.ndarray):
    index.add(embedded_chunks)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 24655.31it/s]


In [ ]:
def query_enhancer(query : str) -> str:
    messages = [
        SystemMessage("""You are an advanced query-expansion and query-rewriting engine for a RAG system. Your job is to take a raw user query and transform it into an optimized search query or a set of sub-queries for a vector database.
Follow these rules strictly:
1. Identify the core intent, entities, and implicit context of the user's question.
2. Add relevant synonyms, domain-specific terminology, and clarifying keywords to bridge vocabulary gaps.
3. If the question is complex or multi-faceted, break it down into distinct, focused sub-queries.
4. Output ONLY the enhanced search query or the list of sub-queries. Do not include conversational filler, explanations, or introductory text.
    """),
    HumanMessage(query)
    ]

    qe_llm = llm.with_structured_output(QueryEnhancerOutput)

    result = qe_llm.invoke(messages)
    return result.query

def search_chunks(index : faiss.IndexFlatIP, query : str, top_k : int = 10):
    embedded_query = embedding_model.encode([query])
    dists, indices = index.search(embedded_query, top_k)

    return indices[0]

def answering(chunks : list[Document], indices : list[int], query : str):
    llm_answer = llm
    knowledge = "\n".join([chunks[i].page_content for i in indices])

    messages = [
        SystemMessage("Answer based on the knowledge that has been given to you. If you don't get the answer from the knowledge, say you don't know. Make your answer as concise as possible"),
        SystemMessage(f"""
        Knowledge:
        {knowledge}
        """),
        HumanMessage(query)
    ]

    result = llm_answer.invoke(messages)
    return result


## Indexing

### Extraction

In [34]:
docs = extraction(file_path)

for i in docs[:3]:
    print(i)

[INFO] 2026-09-26 13:11:00,118 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-26 13:11:00,121 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-26 13:11:00,146 [RapidOCR] download_file.py:60: File exists and is valid: C:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-26 13:11:00,148 [RapidOCR] main.py:50: Using C:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-26 13:11:00,738 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-26 13:11:00,740 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-26 13:11:00,744 [RapidOCR] download_file.py:60: File exists and is valid: C:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-26 13:11:00,745 [RapidOCR] main.py:50: Using C:\imar6teen\projec

page_content='Matthijs Douze FAIR, Meta

## THE FAISS LIBRARY

Alexandr Guzhva Zilliz Chengqi Deng DeepSeek

Jeff Johnson FAIR, Meta Gergely Szilvasy FAIR, Meta Pierre-Emmanuel Mazar´e FAIR, Meta

Maria Lomeli FAIR, Meta Lucas Hosseini Skip Labs Herv´e J´egou FAIR, Meta

## Abstract

Vector databases typically manage large collections of embedding vectors. As AI applications are growing rapidly, the number of embeddings that need to be stored and indexed is increasing. The Faiss library is dedicated to vector similarity search, a core functionality of vector databases. Faiss is a toolkit of indexing methods and related primitives used to search, cluster, compress and transform vectors. This paper describes the trade-offs in vector search and the design principles of Faiss in terms of structure, approach to optimization and interfacing. We benchmark key features of the library and discuss a few selected use cases to highlight its broad applicability.

## 1 Introduction

The emergence of

### Chunking

In [35]:
chunks = chunk_fn(docs)

In [36]:
for chunk in chunks:
    print(chunk.page_content)
    break

Matthijs Douze FAIR, Meta

## THE FAISS LIBRARY

Alexandr Guzhva Zilliz Chengqi Deng DeepSeek

Jeff Johnson FAIR, Meta Gergely Szilvasy FAIR, Meta Pierre-Emmanuel Mazar´e FAIR, Meta

Maria Lomeli FAIR, Meta Lucas Hosseini Skip Labs Herv´e J´egou FAIR, Meta

## Abstract


### Embedding

In [8]:
embedded_chunks = embedding_fn([i.page_content for i in chunks])

### Storing

In [9]:
store_chunks(embedded_chunks)

## Retrieval

In [21]:
query = "Who is the author of this paper?"
enhanced_query = query_enhancer(query)

top_indices = search_chunks(index, enhanced_query, 10)

answer = answering(chunks, top_indices, query)

In [22]:
print(answer.content)

I don't know.


# GraphRAG

In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_docling.loader import DoclingLoader, ExportType, Document
from sentence_transformers import SentenceTransformer
import faiss
import networkx as nx

file_path = os.path.join("..", "assets", "example3.pdf")

def extraction(file_path):
    loader = DoclingLoader(
        file_path = file_path,
        export_type = ExportType.MARKDOWN
    )

    return loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def chunk_fn(docs : list[Document]) -> list[Document]:
    chunks = text_splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["order"] = i 
    return chunks

ext_ent_rels_llm = llm.with_structured_output(ExtractorOutput)

def extract_entities_realtionships(chunks : list[Document]) -> list[ExtractorOutput]:
    sys_message = SystemMessage("""You are an information extraction system. Your task is to extract entities and relationships from the provided text according to the schema below.

## 1. Entity Types

You may extract only these entity types:

* `PERSON` — A human individual.
* `ORGANIZATION` — A company, institution, team, government body, group, or other organized entity.
* `LOCATION` — A geographic place, building, address, city, country, region, or other physical location.
* `OBJECT` — A physical or digital thing, product, document, device, artifact, or other identifiable object.
* `EVENT` — A named or identifiable occurrence, activity, meeting, transaction, competition, incident, etc.
* `CONCEPT` — An abstract idea, topic, theory, subject, technology, methodology, policy, or other non-physical concept.

## 2. Relationship Types

You may extract only these relationship types.

### Social / Ownership

* `FOLLOWS` — PERSON/ORGANIZATION follows another PERSON/ORGANIZATION.
* `FRIENDS_WITH` — Two entities are explicitly described as friends.
* `OWNS` — An entity owns another entity or object.
* `BOUGHT` — An entity purchased another entity or object.

### Organizational

* `WORKS_AT` — A PERSON works for an ORGANIZATION.
* `PART_OF` — An entity is a component, member, division, subgroup, or constituent of another entity.
* `LOCATED_IN` — An entity is physically located in a LOCATION or larger geographic area.
* `CHILD_OF` — An entity is explicitly identified as the child of another entity. Normally applies to PERSON entities unless the text clearly indicates another usage.

### Interaction

* `CREATED_BY` — An OBJECT, EVENT, or other entity was created by another entity.
* `READ` — An entity read a document, book, article, or other readable OBJECT.
* `LIKES` — An entity explicitly likes another entity or object.
* `DEPENDS_ON` — An entity, process, system, concept, or event depends on another entity or concept.

## 3. Extraction Rules

1. Extract only information explicitly stated or strongly entailed by the text.
2. Do not infer relationships based solely on common knowledge, assumptions, stereotypes, or world knowledge.
3. Do not invent entities or relationships.
4. Preserve the entity's surface form from the text whenever possible.
5. If the same entity is mentioned multiple times, resolve the mentions to the same entity when the text clearly indicates they refer to the same thing.
6. Do not create an entity merely because it is implied by a relationship.
7. Extract relationships only when the relationship type matches the defined semantics.
8. Relationship direction matters. Preserve the correct subject → relationship → object direction.
9. If a relationship is ambiguous, do not extract it.
10. If an entity could belong to multiple types, select the most specific appropriate type based on the text.
11. Do not extract relationship types outside the allowed schema.
12. Do not extract entity types outside the allowed schema.
13. Multiple relationships between the same pair of entities are allowed when they are independently supported by the text.
14. Avoid duplicate entities and duplicate relationships.
15. An entity may participate in multiple relationships.
16. Pronouns such as "he", "she", "they", "it", and "this organization" should be resolved to their corresponding entities when the reference is unambiguous.
17. Do not assume that being located in a place means an entity is part of that place's organization or vice versa.
18. `WORKS_AT` should be used only when employment, work, or an equivalent professional relationship is explicitly stated.
19. `OWNS` and `BOUGHT` are distinct:

* `OWNS` represents an ownership relationship.
* `BOUGHT` represents a purchase/acquisition event.
  Do not automatically convert `BOUGHT` into `OWNS` unless the text explicitly establishes current ownership.

20. `CREATED_BY` should represent authorship, creation, production, development, or equivalent creation relationships. Do not use it merely because an entity is associated with another entity.
21. `READ` should only be extracted when the text indicates that an entity actually read or consumed the relevant readable object.
22. `LIKES` requires an explicit indication of preference, appreciation, or liking.
23. `DEPENDS_ON` requires an actual dependency. Mere association or usage is insufficient.

## 4. Entity Normalization

Use a canonical name when the text clearly provides one.

Examples:

* "Elon Musk" → `PERSON`
* "Musk" → resolve to "Elon Musk" when the context clearly establishes this.
* "Google" → `ORGANIZATION`
* "San Francisco" → `LOCATION`
* "iPhone 15" → `OBJECT`
* "the 2024 Olympics" → `EVENT`
* "machine learning" → `CONCEPT`

Do not normalize an entity into a different name unless the equivalence is clear from the text.

## 5. Relationship Direction

Always represent relationships as:

`subject --RELATIONSHIP--> object`

Examples:

* "Alice works at Acme."
  → Alice `WORKS_AT` Acme

* "Bob follows Alice."
  → Bob `FOLLOWS` Alice

* "Acme owns Twitter."
  → Acme `OWNS` Twitter

* "Alice read the report."
  → Alice `READ` the report

* "The application depends on PostgreSQL."
  → The application `DEPENDS_ON` PostgreSQL

Do not reverse the direction.

## 6. Evidence

Every extracted entity and relationship must be supported by the input text.

For relationships, include the text span or a concise evidence phrase that supports the relationship.

If the evidence is insufficient, do not extract the relationship.

## 7. Output Format

Return valid JSON only. Do not include explanations, markdown, comments, or additional text.

Use the following structure:

{
"entities": [
{
"id": "E1",
"name": "entity name",
"type": "PERSON"
}
],
"relationships": [
{
"source": "E1",
"type": "WORKS_AT",
"target": "E2",
"evidence": "Alice works at Acme."
}
]
}

Entity IDs must be unique within the response.

Relationship `source` and `target` must reference valid entity IDs.

## 8. Empty Results

If no entities are found, return:

{
"entities": [],
"relationships": []
}

If entities are found but no relationships are supported, return the entities and an empty relationships array.

## 9. Important Constraint

The goal is high-precision extraction rather than speculation.

When uncertain, prefer omission over invention.

""")
    results : list[ExtractorOutput] = []
    i = 0
    for chunk in chunks:
        if i > 10:
            break
        messages = [
            sys_message,
            HumanMessage(chunk.page_content)
        ]

        result = ext_ent_rels_llm.invoke(messages)
        results.append(result)
        i+=1
        
    return results

def entity_resolution(ent_rel_collection : list[ExtractorOutput], similarity_threshold : float = 0.82) -> ExtractorOutput:
    """Refer to this [paper](https://arxiv.org/html/2402.07630v3) and this [code](https://github.com/157114/Denoise/tree/main) for handling duplicated entities and relationships.
    Args:
        ent_rel_collection (list[ExtractorOutput]): this is collection of entities and relationships
        similarity_threshold (float): cosine similarity threshold for merging entities

    Returns:
        ExtractorOutput: The entities and relationships after done entity resolution 
    """
    from collections import defaultdict, Counter
    from sklearn.cluster import KMeans
    import numpy as np

    if not ent_rel_collection:
        return ExtractorOutput(entities=[], relationships=[])

    # -------------------------------------------------------------
    # 0. Flatten & Normalize Input Entities and Relationships
    # -------------------------------------------------------------
    # Each entity extracted from a chunk has a local ID (e.g. 'E1').
    # We assign a globally unique key: (chunk_idx, local_id).
    node_records = []
    chunk_local_to_key = {}
    chunk_name_to_key = {}

    for chunk_idx, item in enumerate(ent_rel_collection):
        entities = item.entities if hasattr(item, 'entities') else item.get('entities', [])
        chunk_local_to_key[chunk_idx] = {}
        chunk_name_to_key[chunk_idx] = {}

        for ent_idx, ent in enumerate(entities):
            ent_dict = ent if isinstance(ent, dict) else (ent.dict() if hasattr(ent, 'dict') else ent.__dict__)
            local_id = str(ent_dict.get('id', f'E{ent_idx + 1}')).strip()
            name = str(ent_dict.get('name', '')).strip()
            ent_type = str(ent_dict.get('type', 'CONCEPT')).strip().upper()

            if not name:
                continue

            node_key = f"c{chunk_idx}_{local_id}"
            node_records.append({
                'key': node_key,
                'chunk_idx': chunk_idx,
                'local_id': local_id,
                'name': name,
                'type': ent_type,
                'raw': ent_dict
            })
            chunk_local_to_key[chunk_idx][local_id] = node_key
            chunk_name_to_key[chunk_idx][name.lower()] = node_key

    if not node_records:
        return ExtractorOutput(entities=[], relationships=[])

    key_to_record = {r['key']: r for r in node_records}
    all_keys = [r['key'] for r in node_records]

    # Resolve embedding model (from global scope if defined, e.g. BAAI/bge-m3)
    emb_model = globals().get('embedding_model', None)
    key_to_embedding = {}
    if emb_model is not None:
        texts = [f"Entity: {r['name']}. Type: {r['type']}" for r in node_records]
        embeddings = emb_model.encode(texts, show_progress_bar=False, normalize_embeddings=True)
        for r, emb in zip(node_records, embeddings):
            key_to_embedding[r['key']] = emb

    # -------------------------------------------------------------
    # Blocking
    # Reference: DEG-RAG (denoise_kg/cluster.py)
    # - Group entities by entity_type (entity_block)
    # - For large blocks, sub-cluster using KMeans to avoid O(N^2) comparisons
    # -------------------------------------------------------------
    type_blocks = defaultdict(list)
    for r in node_records:
        type_blocks[r['type']].append(r['key'])

    candidate_blocks = []
    min_cluster_size_kmeans = 15

    for etype, keys in type_blocks.items():
        if len(keys) <= 1:
            candidate_blocks.append(keys)
            continue

        # If block is large and embeddings are available, run KMeans sub-clustering
        if len(keys) >= min_cluster_size_kmeans and key_to_embedding:
            k = max(2, min(len(keys) // 5, 8))
            sub_embs = np.array([key_to_embedding[k_id] for k_id in keys])
            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
            labels = kmeans.fit_predict(sub_embs)
            sub_clusters = defaultdict(list)
            for k_id, label in zip(keys, labels):
                sub_clusters[label].append(k_id)
            for sub_keys in sub_clusters.values():
                candidate_blocks.append(sub_keys)
        else:
            candidate_blocks.append(keys)

    # -------------------------------------------------------------
    # Matching & Grouping
    # Reference: DEG-RAG (denoise_kg/similar_percentage.py & merge.py)
    # - Exact normalized string matching (high precision)
    # - Cosine similarity matching within candidate blocks
    # - Disjoint Set Union (Union-Find) to form entity groups
    # -------------------------------------------------------------
    parent = {k: k for k in all_keys}

    def find_set(v):
        if parent[v] != v:
            parent[v] = find_set(parent[v])
        return parent[v]

    def unite_sets(a, b):
        ra, rb = find_set(a), find_set(b)
        if ra != rb:
            parent[rb] = ra

    # 1. Exact normalized name matching across all chunks
    name_to_keys = defaultdict(list)
    for r in node_records:
        norm_name = r['name'].lower().strip()
        name_to_keys[norm_name].append(r['key'])

    for same_name_keys in name_to_keys.values():
        if len(same_name_keys) > 1:
            for k_other in same_name_keys[1:]:
                unite_sets(same_name_keys[0], k_other)

    # 2. Pairwise cosine similarity inside candidate blocks
    for block in candidate_blocks:
        if len(block) < 2:
            continue

        for i in range(len(block)):
            key1 = block[i]
            for j in range(i + 1, len(block)):
                key2 = block[j]

                if find_set(key1) == find_set(key2):
                    continue

                if key1 in key_to_embedding and key2 in key_to_embedding:
                    # Normalized vectors dot product = cosine similarity
                    sim = float(np.dot(key_to_embedding[key1], key_to_embedding[key2]))
                    if sim >= similarity_threshold:
                        unite_sets(key1, key2)

    # Group node records by canonical root
    cluster_groups = defaultdict(list)
    for k in all_keys:
        root = find_set(k)
        cluster_groups[root].append(key_to_record[k])

    # -------------------------------------------------------------
    # Merging & Linking
    # Reference: DEG-RAG (denoise_kg/merge.py & Core/Utils/MergeSum.py)
    # - Select canonical name (most frequent, longest as tie-breaker)
    # - Select canonical type (majority vote)
    # - Remap relationship sources and targets to canonical entity IDs
    # - Deduplicate relationships, merge evidence and track weights
    # -------------------------------------------------------------
    canonical_entities = []
    key_to_canonical_id = {}

    for new_idx, (root, members) in enumerate(cluster_groups.items(), start=1):
        canonical_id = f"E{new_idx}"

        # Canonical Name: most frequent name across chunks, longest as tie-breaker
        name_counts = Counter(m['name'] for m in members)
        sorted_names = sorted(name_counts.keys(), key=lambda x: (name_counts[x], len(x)), reverse=True)
        canonical_name = sorted_names[0]

        # Canonical Type: most common type
        type_counts = Counter(m['type'] for m in members)
        canonical_type = type_counts.most_common(1)[0][0]

        for m in members:
            key_to_canonical_id[m['key']] = canonical_id

        canonical_entities.append({
            'id': canonical_id,
            'name': canonical_name,
            'type': canonical_type
        })

    # Relink and merge relationships
    rel_map = {}  # (src_canonical, rel_type, tgt_canonical) -> dict

    for chunk_idx, item in enumerate(ent_rel_collection):
        relationships = item.relationships if hasattr(item, 'relationships') else item.get('relationships', [])

        for rel in relationships:
            rel_dict = rel if isinstance(rel, dict) else (rel.dict() if hasattr(rel, 'dict') else rel.__dict__)
            src_raw = str(rel_dict.get('source', '')).strip()
            rel_type = str(rel_dict.get('type', 'RELATED_TO')).strip().upper()
            tgt_raw = str(rel_dict.get('target', '')).strip()
            evidence = str(rel_dict.get('evidence', '')).strip()

            src_key = chunk_local_to_key.get(chunk_idx, {}).get(src_raw) or chunk_name_to_key.get(chunk_idx, {}).get(src_raw.lower())
            tgt_key = chunk_local_to_key.get(chunk_idx, {}).get(tgt_raw) or chunk_name_to_key.get(chunk_idx, {}).get(tgt_raw.lower())

            if not src_key or not tgt_key:
                continue

            src_canonical = key_to_canonical_id.get(src_key)
            tgt_canonical = key_to_canonical_id.get(tgt_key)

            if not src_canonical or not tgt_canonical:
                continue

            # Remove self-loops created by entity resolution
            if src_canonical == tgt_canonical:
                continue

            edge_key = (src_canonical, rel_type, tgt_canonical)
            if edge_key not in rel_map:
                rel_map[edge_key] = {
                    'source': src_canonical,
                    'type': rel_type,
                    'target': tgt_canonical,
                    'evidence': set([evidence]) if evidence else set(),
                    'weight': 1
                }
            else:
                if evidence:
                    rel_map[edge_key]['evidence'].add(evidence)
                rel_map[edge_key]['weight'] += 1

    canonical_relationships = []
    for edge in rel_map.values():
        ev_list = sorted(list(edge['evidence']))
        canonical_relationships.append({
            'source': edge['source'],
            'type': edge['type'],
            'target': edge['target'],
            'evidence': ' ; '.join(ev_list) if ev_list else '',
            'weight': edge['weight']
        })

    return ExtractorOutput(
        entities=canonical_entities,
        relationships=canonical_relationships
    )

def build_kg(ent_rel : ExtractorOutput):
    """Use this for building knowledge graph based on list of entities and relationships.
    Args:
        ent_rel (ExtractorOutput): collection of entities and relationships after from entity resolution
    """
    G = nx.DiGraph()

    ents, rels = ent_rel.entities, ent_rel.relationships

    for i in ents:
        G.add_node(i["id"], name=i["name"], type=i["type"])
    for i in rels:
        G.add_edge(i["source"], i["target"], type=i["type"], evidence=i["evidence"], weight=i["weight"])
        
    return G

embedding_model = SentenceTransformer("BAAI/bge-m3")

def embedding_fn(chunk_page_content : list[str]):
    return embedding_model.encode(chunk_page_content)

d = 1024 # depending on the embedding model dimension
#* IndexFlatIP stores normalized vector, but BAAI/bge-m3 output is a normalized vector
#* so, doesn't need to normalize the output vector again
index_nodes = faiss.IndexFlatIP(d)
index_edges = faiss.IndexFlatIP(d)

def store_chunks(embedded_chunks : np.ndarray):
    index.add(embedded_chunks)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 28697.95it/s]


In [55]:
def query_enhancer(query : str) -> str:
    messages = [
        SystemMessage("""You are an advanced query-expansion and query-rewriting engine for a RAG system. Your job is to take a raw user query and transform it into an optimized search query or a set of sub-queries for a vector database.
Follow these rules strictly:
1. Identify the core intent, entities, and implicit context of the user's question.
2. Add relevant synonyms, domain-specific terminology, and clarifying keywords to bridge vocabulary gaps.
3. If the question is complex or multi-faceted, break it down into distinct, focused sub-queries.
4. Output ONLY the enhanced search query or the list of sub-queries. Do not include conversational filler, explanations, or introductory text.
    """),
    HumanMessage(query)
    ]

    qe_llm = llm.with_structured_output(QueryEnhancerOutput)

    result = qe_llm.invoke(messages)
    return result.query

def search_ents_rels(index_nodes : faiss.IndexFlatIP, index_edges: faiss.IndexFlatIP, query : str, top_k : int = 10):
    emb_query = embedding_fn([query])
    _, node_indices = index_nodes.search(emb_query, top_k)
    _, edge_indices = index_edges.search(emb_query, top_k)
    return node_indices, edge_indices

def subgraph_construction():
    pass

def answering(chunks : list[Document], indices : list[int], query : str):
    llm_answer = llm
    knowledge = "\n".join([chunks[i].page_content for i in indices])

    messages = [
        SystemMessage("Answer based on the knowledge that has been given to you. If you don't get the answer from the knowledge, say you don't know. Make your answer as concise as possible"),
        SystemMessage(f"""
        Knowledge:
        {knowledge}
        """),
        HumanMessage(query)
    ]

    result = llm_answer.invoke(messages)
    return result


## Indexing

In [8]:
chunks = chunk_fn(extraction(file_path))

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-10-01 17:25:06,437 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-10-01 17:25:06,478 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-10-01 17:25:06,520 [RapidOCR] download_file.py:60: File exists and is valid: C:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-10-01 17:25:06,522 [RapidOCR] main.py:50: Using C:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-10-01 17:25:07,152 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-10-01 17:25:07,154 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-10-01 17:25:07,167 [RapidOCR] download_file.p

In [9]:
tes = extract_entities_realtionships(chunks)

In [10]:
tis = entity_resolution(tes)

In [41]:
ent_rel_per_chunk = tes
er = tis

In [71]:
er.relationships[0]

{'source': 'E1',
 'type': 'WORKS_AT',
 'target': 'E2',
 'evidence': 'Matthijs Douze FAIR ; Matthijs Douze FAIR, Meta',
 'weight': 2}

In [ ]:
emb_nodes = embedding_fn([f"name: {i['name']}, type: {i['type']}" for i in er.entities])
emb_edges = embedding_fn([f"type: {i['type']}, evidence: {i['evidence']}, weight: {i['weight']}" for i in er.relationships])
index_nodes.add(emb_nodes)
index_edges.add(emb_edges)
kg = build_kg(er)

## Retrieval

In [59]:
query = "Who is the author of this paper?"
# query = query_enhancer(query)

In [62]:
node_indices, edge_indices = search_ents_rels(index_nodes, index_edges, query, 10)

In [77]:
for i in node_indices[0]:
    print(er.entities[i])
for i in edge_indices[0]:
    print(er.relationships[i])

{'id': 'E4', 'name': 'Alexandr Guzhva', 'type': 'PERSON'}
{'id': 'E6', 'name': 'Zilliz Chengqi Deng', 'type': 'PERSON'}
{'id': 'E19', 'name': 'self-supervision', 'type': 'CONCEPT'}
{'id': 'E10', 'name': 'Maria Lomeli', 'type': 'PERSON'}
{'id': 'E15', 'name': 'selected use cases', 'type': 'CONCEPT'}
{'id': 'E16', 'name': 'embeddings', 'type': 'CONCEPT'}
{'id': 'E18', 'name': 'multi-modal text-image', 'type': 'CONCEPT'}
{'id': 'E11', 'name': 'Lucas Hosseini', 'type': 'PERSON'}
{'id': 'E13', 'name': 'Hervé Jégou', 'type': 'PERSON'}
{'id': 'E21', 'name': 'distance metric', 'type': 'CONCEPT'}
{'source': 'E4', 'type': 'CREATED_BY', 'target': 'E3', 'evidence': 'Alexandr Guzhva Zilliz Chengqi Deng DeepSeek', 'weight': 1}
{'source': 'E10', 'type': 'CREATED_BY', 'target': 'E3', 'evidence': 'Maria Lomeli FAIR, Meta', 'weight': 1}
{'source': 'E1', 'type': 'CREATED_BY', 'target': 'E3', 'evidence': 'Matthijs Douze FAIR, Meta', 'weight': 1}
{'source': 'E4', 'type': 'WORKS_AT', 'target': 'E5', 'eviden